# 09 Frontend Validation

Smoke-check the Streamlit app structure, page modules, and required runtime artifacts without launching a long-running UI server.

## Setup

In [1]:
import ast
import os
import sys
import json
import py_compile
from pathlib import Path

import pandas as pd
from IPython.display import display

sys.path.append(os.path.abspath('..'))
from src.utils import load_config

config = load_config('../configs/config.yaml')
paths = config['paths']
project_root = Path('..').resolve()
processed_dir = project_root / paths['processed_data_dir']
logs_dir = project_root / 'logs'
outputs_dir = project_root / 'outputs'
models_dir = project_root / 'models'
app_path = project_root / 'app.py'
pages_dir = project_root / 'pages'
page_files = sorted(pages_dir.glob('*.py'))


## Syntax and Import Smoke Checks

In [2]:
validation_rows = []
for path in [app_path] + page_files:
    row = {'file': str(path), 'syntax_ok': False, 'ast_ok': False, 'error': ''}
    try:
        py_compile.compile(str(path), doraise=True)
        row['syntax_ok'] = True
    except Exception as exc:
        row['error'] = f'py_compile: {exc}'

    try:
        ast.parse(path.read_text(encoding='utf-8'))
        row['ast_ok'] = True
    except Exception as exc:
        row['error'] = (row['error'] + ' | ' if row['error'] else '') + f'ast: {exc}'

    validation_rows.append(row)

display(pd.DataFrame(validation_rows))


,file,syntax_ok,ast_ok,error
0,D:\Desertation\Code_v3\flight-disruption-predi...,True,True,
1,D:\Desertation\Code_v3\flight-disruption-predi...,True,True,
2,D:\Desertation\Code_v3\flight-disruption-predi...,True,True,
3,D:\Desertation\Code_v3\flight-disruption-predi...,True,True,
4,D:\Desertation\Code_v3\flight-disruption-predi...,True,True,
5,D:\Desertation\Code_v3\flight-disruption-predi...,True,True,
6,D:\Desertation\Code_v3\flight-disruption-predi...,True,True,
7,D:\Desertation\Code_v3\flight-disruption-predi...,True,True,


## Artifact Readiness Matrix

In [3]:
artifact_checks = {
    'ml_dataset': processed_dir / paths['ml_dataset_file'],
    'trajectory_features': processed_dir / paths['features_file'],
    'trajectories': processed_dir / paths['trajectories_file'],
    'bts_combined': processed_dir / 'bts_combined.parquet',
    'eurocontrol_combined': processed_dir / 'eurocontrol_combined.parquet',
    'quality_gates_report': logs_dir / 'quality_gates_report.json',
    'weather_coverage': logs_dir / 'weather_coverage.json',
    'label_distribution': logs_dir / 'label_distribution.json',
    'model_comparison': logs_dir / 'model_comparison.json',
    'roc_curves': outputs_dir / 'roc_curves.png',
    'shap_summary': outputs_dir / 'shap_summary.png',
    'drift_report': logs_dir / 'drift_report.json',
    'scaler': models_dir / 'scaler.pkl',
    'imputer': models_dir / 'imputer.pkl',
    'label_encoder': models_dir / 'label_encoder.pkl',
}

artifact_df = pd.DataFrame({
    'artifact': list(artifact_checks.keys()),
    'path': [str(path) for path in artifact_checks.values()],
    'exists': [path.exists() for path in artifact_checks.values()],
})
display(artifact_df)


,artifact,path,exists
0,ml_dataset,D:\Desertation\Code_v3\flight-disruption-predi...,True
1,trajectory_features,D:\Desertation\Code_v3\flight-disruption-predi...,True
2,trajectories,D:\Desertation\Code_v3\flight-disruption-predi...,True
3,bts_combined,D:\Desertation\Code_v3\flight-disruption-predi...,True
4,eurocontrol_combined,D:\Desertation\Code_v3\flight-disruption-predi...,True
5,quality_gates_report,D:\Desertation\Code_v3\flight-disruption-predi...,True
6,weather_coverage,D:\Desertation\Code_v3\flight-disruption-predi...,True
7,label_distribution,D:\Desertation\Code_v3\flight-disruption-predi...,True
8,model_comparison,D:\Desertation\Code_v3\flight-disruption-predi...,False
9,roc_curves,D:\Desertation\Code_v3\flight-disruption-predi...,False


## Page Readiness Summary

In [4]:
page_requirements = {
    'Pipeline Overview': ['quality_gates_report'],
    'Data Explorer': ['ml_dataset', 'bts_combined', 'eurocontrol_combined'],
    'Trajectory Map': ['trajectories'],
    'Feature Analysis': ['ml_dataset'],
    'Model Performance': ['model_comparison', 'roc_curves'],
    'Predictions Explorer': ['scaler', 'imputer', 'label_encoder'],
    'Data Quality': ['quality_gates_report', 'weather_coverage'],
}

artifact_exists = {name: path.exists() for name, path in artifact_checks.items()}
summary_rows = []
for page_name, required in page_requirements.items():
    missing = [item for item in required if not artifact_exists.get(item, False)]
    summary_rows.append({
        'page': page_name,
        'required_artifacts': ', '.join(required),
        'missing_artifacts': ', '.join(missing),
        'ready': len(missing) == 0,
    })

display(pd.DataFrame(summary_rows))


,page,required_artifacts,missing_artifacts,ready
0,Pipeline Overview,quality_gates_report,,True
1,Data Explorer,"ml_dataset, bts_combined, eurocontrol_combined",,True
2,Trajectory Map,trajectories,,True
3,Feature Analysis,ml_dataset,,True
4,Model Performance,"model_comparison, roc_curves","model_comparison, roc_curves",False
5,Predictions Explorer,"scaler, imputer, label_encoder",,True
6,Data Quality,"quality_gates_report, weather_coverage",,True


## Optional Launch Command

In [5]:
print('Run this manually when you want the live frontend:')
print('streamlit run app.py')


Run this manually when you want the live frontend:
streamlit run app.py
